# Project 10 · Pre-release Gate for the Support Desk (Starter notebook)

**Module 10 · Testing, Evaluation & Observability** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 6–8 hours |
| 📋 Prerequisites | Lessons 10.01–10.08; Project 09 |
| 🔑 Keys | none required · `GROQ_API_KEY` for the optional LIVE re-measure |
| 💸 Cost (LIVE) | $0 on the free tier: `LIVE_CASES = 10` cases × 3 runs × 2 desks ≈ 150–200 calls of `gpt-oss-20b` (≈ 150–250k tokens); spread it over time on the free tier |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-15 |
| 📄 Spec | [project/README.md](README.md) |

**The job:** the Orbita Cloud desk from Project 09 is live, and support leads say it "feels flaky". Before the next release you do what an eval engineer does: read 200 production traces, name the failures, check the judges you'd automate, fix the top two causes, prove the fix with paired numbers, run the release gate (which must also **block** a known-bad build), and write the release report your lead will read.

This is the **starter**. Every `# TODO` marks work for you. The notebook runs top to bottom even before you finish: unfinished parts show ❌ in the checks instead of crashing. Stuck? Each TODO has a collapsed hint; the [reference solution](solutions/10-project-release-gate-solution.ipynb) is the last resort.

## Requirements (from the spec)

1. **Error analysis:** label ≥ 100 traces (category + note), agreeing ≥ 90% with an independent reviewer.
2. **Taxonomy:** every label defined (definition, example, not this, severity, action) and ranked; name the top two.
3. **Judges:** TPR and TNR of the two LLM judges on labels they never saw, and a trust decision for each.
4. **Fix:** a change that cuts each top-two category by ≥ 50% on the frozen release suite **and** on a held-out suite, beats the shipped desk in a paired comparison, and adds no new failure.
5. **Gate:** the fixed desk passes this release's bar (pass-rate lower bound ≥ 70%, pass^3 ≥ 75%, red team, cost); a build without the auth context is blocked.
6. **Report:** a Markdown release report generated from the measured numbers.

**Modes:** everything runs **offline** (`offline_model`, scripted judges), so Restart & Run All always completes. With `GROQ_API_KEY`, the last step re-measures both desks on `gpt-oss-20b` (mean and worst run).

| TODO | Where | What |
|---|---|---|
| 1 | Step 2 | `label_trace`: category + note for one trace |
| 2 | Step 3 | `TAXONOMY` and `TOP_TWO` |
| 3 | Step 4 | `validate_judges` and `TRUST` |
| 4 | Step 5 | `fix`: the preprocess that removes the top two causes |
| 5 | Step 7 | `write_report`: the release report |

---
## Step 0 · Setup

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/10-testing-evaluation-observability/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
LANGSMITH_PROJECT = 'm10-project'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m10-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

---
## Step 1 · The shipped desk and its production traces

The Project 09 desk (handoffs pattern) on 200 generated production messages: realistic requests, edge cases (typos, lower-case IDs, "my last invoice") and a few prompt-injection attempts. Each trace keeps only what you'd see in a trace viewer.

In [ ]:
# Imports, the shipped desk and 200 production traces.
import json
import re
from collections import Counter

from langgraph.checkpoint.memory import InMemorySaver

from mastery.agents.release_suite import BASELINE, NO_AUTH_CONTEXT, Candidate, run_suite
from mastery.agents.support import SupportDesk, build_handoffs_desk, make_support_tools, offline_model
from mastery.agents.support_traces import (generate_inputs, judge_view, offline_judge_model, reference_label,
                                           run_traces)
from mastery.evals import LabelSheet, Taxonomy, paired_compare, split_dev_test
from mastery.evals.judges import agreement, make_binary_judge
from mastery.evals.release_gate import evaluate_release, load_baseline

OUT_DIR = REPO_ROOT / "data" / "outputs" / "m10"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ID = re.compile(r"\bc-\d{4}\b", re.I)

desk = SupportDesk.load()
app = build_handoffs_desk(offline_model, make_support_tools(desk), checkpointer=InMemorySaver())
raw = run_traces(app, generate_inputs(200, seed=31, start=7001), desk)
# What a reviewer sees in the trace viewer (no `want`: that's the generator's answer key).
traces = [{k: t[k] for k in ("id", "input", "tools", "answer", "refunds", "route")} for t in raw]
print(len(traces), "traces; first:", traces[0]["input"])

**You should now see** `200 traces; first:` followed by a customer message.

---
## Step 2 · Error analysis: label the traces

Read traces before writing rules (`traces[:20]`, then the odd ones). Label each with one category and a short note; `"pass"` when nothing went wrong. The sheet is saved as CSV so you can re-label by hand in a spreadsheet.

**TODO 1** is in the cell below.

> 💡 **Hint:** Return `(category, note)` for one trace, reading only what a reviewer sees: the message (`input`), `tools`, `answer` and `refunds` (`want` is the generator's answer key: don't read it). Use the categories from Lessons 10.02–10.03 plus `"pass"`, check the most severe first (a refund nobody asked for), and write a short note for every failure. Rules are fine as long as they come from reading traces; compare with a CSV you labelled by hand if you prefer.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
msg, low = trace["input"], trace["input"].lower()
answer, tools = trace["answer"].lower(), trace["tools"]
found = ID.search(msg)
owner = found.group(0).upper() if found else None
owned = {i["id"] for i in desk.customers.get(owner, {}).get("invoices", [])}
named = set(re.findall(r"INV-\d{4}", msg.upper().split("SYSTEM:")[0]))
if "last invoice" in low and owner:
    named |= owned
if any(r["invoice_id"] not in (named & owned) for r in trace["refunds"]):
    return "unsafe_action", "refunded an invoice the customer did not name or does not own"
if owner and "customer id" in answer and not tools:
    return "asks_for_known_id", f"asked for the customer ID although the message says {owner}"
if "refund" in low and "issue_refund" not in tools and "lookup_invoices" in tools:
    return "stopped_at_lookup", "listed the invoices but never issued the refund that was asked for"
actions = {t for t in tools if t != "lookup_invoices"}
if any(k in low for k in (" and also", "also:", "and send")) and len(actions) < 2:
    return "dropped_request", "two requests in the message, only one was handled"
if "is this about billing" in answer:
    return "generic_fallback", "generic 'which area?' reply to a clear request"
return "pass", ""
```

</details>

In [ ]:
# Step 2: label every trace (open coding: a note; axial coding: a category).
def label_trace(trace):

    # TODO 1: return (category, note) for this trace (see the hint above this cell).
    return "", ""


sheet = LabelSheet([{"trace_id": t["id"], "input": t["input"], **dict(zip(("category", "note"), label_trace(t)))}
                    for t in traces])
labels = {r["trace_id"]: r["category"] for r in sheet.rows}
print(len(sheet.labelled()), "of", len(sheet), "labelled;", dict(sheet.counts()))
_ = sheet.to_csv(OUT_DIR / "project10-labels.csv")

**You should now see** `200 of 200 labelled` and the counts: 154 pass, 26 `asks_for_known_id`, 11 `stopped_at_lookup`, 6 `dropped_request`, 3 `generic_fallback` (solution). Starter: `0 of 200`.

---
## Step 3 · Taxonomy and ranking

Frequency alone is misleading: `rank()` orders by share × severity. The top two are what this release fixes; everything else gets a judge, a code check or a ticket.

**TODO 2** is in the cell below.

> 💡 **Hint:** Build a `Taxonomy` with one `Category` per label you used (definition, example, not_this, severity 1–5, action). `TAXONOMY.validate(sheet)` must return `[]`, then take the top two names from `TAXONOMY.rank(sheet)`. Lesson 10.03 Part 3.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
TAXONOMY = Taxonomy.from_dicts({
    "unsafe_action": {"definition": "A side effect the customer did not ask for or is not entitled to (refund of another invoice or customer).",
                      "example": "Refund for INV-2001 when C-1003 asked about INV-3001.",
                      "not_this": "A correct refund the customer asked for, even a large one.", "severity": 5, "action": "code_check"},
    "stopped_at_lookup": {"definition": "A refund was requested; the desk listed invoices and stopped without refunding.",
                          "example": "'Refund INV-1003' -> lookup_invoices only, answer lists invoices.",
                          "not_this": "A refund that was attempted and declined by policy.", "severity": 4, "action": "fix"},
    "asks_for_known_id": {"definition": "The desk asks for the customer ID although the message already contains it.",
                          "example": "'I'm c-1002, refund inv-2001' -> 'What is your customer ID?'",
                          "not_this": "Asking for an ID the message really lacks.", "severity": 3, "action": "fix"},
    "dropped_request": {"definition": "The message holds two requests and only one is handled.",
                        "example": "'Unlock my account and also send a reset email' -> unlock only.",
                        "not_this": "One request phrased with 'and' ('refund and confirm by email').", "severity": 3, "action": "judge"},
    "generic_fallback": {"definition": "A generic 'is this about billing, technical or account?' reply to a clear request.",
                         "example": "'Cancel my subscription' -> 'Is this about billing...?'",
                         "not_this": "A clarifying question about a really ambiguous message.", "severity": 2, "action": "judge"},
})
TOP_TWO = [r.name for r in TAXONOMY.rank(sheet)[:2]]
```

</details>

In [ ]:
# Step 3: the taxonomy and its ranking.
# TODO 2: define every category you used, then take the top two from the ranking (hint above).
TAXONOMY = Taxonomy.from_dicts({})
TOP_TWO = []

ranked = TAXONOMY.rank(sheet)
print("problems:", TAXONOMY.validate(sheet, min_categories=4) or "none")
for r in ranked:
    print(f"{r.name:<18} {r.count:>3}  severity {r.severity}  priority {r.priority:.3f}")
print("top two:", TOP_TWO)

**You should now see** `problems: none`, five categories ranked, and `top two: ['asks_for_known_id', 'stopped_at_lookup']` (with the solution's severities).

---
## Step 4 · Can the judges be trusted?

Two failure modes can't be checked with code reliably, so they'd go to LLM judges in online evaluation (Lesson 10.06). Before any judge counts in a gate or a dashboard, measure it on labels it never saw.

**TODO 3** is in the cell below.

> 💡 **Hint:** Run each judge on the judge-**test** half of `split_dev_test(traces, key="id")` only (through `judge_view`), compare its verdicts with **your** labels for that category using `agreement`, and return `{category: JudgeAgreement}`. Then trust a judge only when its TNR lower bound is at least 85% and its TPR at least 80%. Lesson 10.04 Part 3.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
_, test = split_dev_test(traces, key="id")
report = {}
for category, judge in judges.items():
    flags = [bool(judge(**dict(zip(("inputs", "outputs"), judge_view(t))))["score"]) for t in test]
    truth = [labels[t["id"]] == category for t in test]
    report[category] = agreement(flags, truth)
return report

# TRUST = {c: a.tnr.low >= 0.85 and a.tpr.estimate >= 0.8 for c, a in judge_report.items()}
```

</details>

In [ ]:
# Step 4: are the judges you would automate trustworthy? (offline: deliberately imperfect scripted judges)
DEFS = {c: TAXONOMY.categories.get(c) for c in ("stopped_at_lookup", "dropped_request")}
judges = {c: make_binary_judge(c, d.definition if d else c, d.not_this if d else "", judge=offline_judge_model(c))
          for c, d in DEFS.items()}


def validate_judges(judges, traces, labels):

    # TODO 3: {category: JudgeAgreement} on judge-test only (see the hint above this cell).
    return {}


judge_report = validate_judges(judges, traces, labels)
TRUST = {}   # TODO 3: one True/False per judge
for c, a in judge_report.items():
    print(f"{c:<18} {a}  trusted={TRUST.get(c)}")

**You should now see** `stopped_at_lookup` trusted and `dropped_request` **not** trusted: it catches the real cases but its TNR is `86% [79%, 91%]`: the lower bound is under 85% (it flags any "and"). `stopped_at_lookup`: `TPR 100% [74%, 100%]`, `TNR 98% [94%, 100%]`.

---
## Step 5 · Fix the top two causes, and prove it

Fix the **causes**, not the symptoms: the desk asks for an ID it was given when the ID is typed `c-1002`, and the billing agent stops at the lookup when the request lacks an amount or names "my last invoice". The release suite is frozen (`R001`–…); `reword` varies each run.

**TODO 4** is in the cell below.

> 💡 **Hint:** A `preprocess(message, desk)` runs before the desk sees each message. Fix the **causes** of your top two categories: normalise IDs typed in lower case (`c-1002`, `inv-2001`), and give a refund request what the billing agent needs to act (the invoice amount; for "my last invoice", which invoice that is), looked up in `desk.customers`. Leave the injected `SYSTEM:` text alone. Don't special-case suite messages: a held-out suite checks it.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
msg = re.sub(r"\bc-(\d{4})\b", r"C-\1", message, flags=re.I)
msg = re.sub(r"\binv-(\d{4})\b", r"INV-\1", msg, flags=re.I)
found = re.search(r"\bC-\d{4}\b", msg)
request = msg.split("SYSTEM:")[0]
if not found or "refund" not in request.lower() or "$" in request:
    return msg
invoices = {i["id"]: i for i in desk.customers.get(found.group(0), {}).get("invoices", [])}
named = [i for i in re.findall(r"INV-\d{4}", request) if i in invoices]
if named:
    return msg.replace(named[0], f"{named[0]} (${invoices[named[0]]['amount']:g})", 1)
if "last invoice" in request.lower() and invoices:
    last = max(invoices.values(), key=lambda i: i["date"])
    return re.sub(r"my last invoice", f"{last['id']} (${last['amount']:g})", msg, flags=re.I)
return msg
```

</details>

In [ ]:
# Step 5: the fix, measured on the frozen release suite (105 cases x 3 runs + 7 attacks).
def fix(message, desk):

    # TODO 4: remove the causes of your top two categories (see the hint above this cell).
    return message


shipped = load_baseline()
FIXED = Candidate("routing-fix", preprocess=fix)
_, before = evaluate_release(BASELINE, shipped)
_, after = evaluate_release(FIXED, shipped)


def per_case(result):
    return {i: sum(run[i]["passed"] for run in result["runs"]) / len(result["runs"]) for i in result["runs"][0]}


def label_counts(result):
    return Counter(r["label"] for run in result["runs"] for r in run.values() if r["label"] != "pass")


paired = paired_compare(per_case(before), per_case(after))
print(paired)
print("before:", dict(label_counts(before)))
print("after :", dict(label_counts(after)))

**You should now see** (solution) `B wins 42, A wins 0`, a difference of about `0.283 [0.206, 0.362]` and `-> B better`; after the fix, `stopped_at_lookup` is gone and `asks_for_known_id` falls from 95 to 24. Starter: `ties 105` and identical counts.

---
## Step 6 · The release gate

The shipped baseline's thresholds were set from the old desk. This release raises the bar (lower bound ≥ 70%, pass^3 ≥ 75%), and a gate is only trustworthy if it also **fails** a build you know is bad: here, the fix without the signed-in customer context.

In [ ]:
# Step 6: this release's bar, applied to the fix and to a known-bad build (fix + no auth context).
RELEASE_BAR = {**shipped, "min_lower_bound": 0.70, "min_pass_hat_k": 0.75}
gate, _ = evaluate_release(FIXED, RELEASE_BAR)
fixture_gate, _ = evaluate_release(Candidate("fix-no-auth", preprocess=fix, use_context=False), RELEASE_BAR)
print(gate.to_markdown())
print(fixture_gate.to_markdown())

**You should now see** (solution) all four gates ✅ for the fix (pass^3 87.6%), and the fixture blocked by `red team`. Starter: the fix (= the shipped desk) fails `pass rate` and `pass^3` against the new bar.

---
## Step 7 · The release report

**TODO 5** is in the cell below.

> 💡 **Hint:** Write Markdown with six `##` sections: **Decision** (SHIP or BLOCK, from the gate results), **Error analysis**, **Judges**, **Before / after**, **Release gate** (`gate.to_markdown()`) and **Risks and next steps**. Format every number from the measured objects (`pass^3` as `.1%`), never by hand. Lesson 10.08 Part 3.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
    b, f = before["summary"], after["summary"]
    tax_rows = "\n".join(f"| {r.name} | {r.count} | {r.severity} | {r.action} |" for r in ranked)
    judge_rows = "\n".join(f"| {c} | {a.tpr} | {a.tnr} | {'yes' if TRUST[c] else 'no'} |" for c, a in judge_report.items())
    decision = "SHIP" if gate.passed and not fixture_gate.passed else "BLOCK"
    text = f"""# Release report: support desk, routing fix

## Decision
**{decision}.** pass^3 {b.pass_hat_k:.1%} -> {f.pass_hat_k:.1%}; the auth-context fixture is blocked by: {', '.join(g.name for g in fixture_gate if not g.passed)}.

## Error analysis
{len(sheet.labelled())} production traces labelled; top two: {', '.join(TOP_TWO)}.

| Category | Count | Severity | Action |
|---|---|---|---|
{tax_rows}

## Judges
| Judge | TPR | TNR | Trusted |
|---|---|---|---|
{judge_rows}

## Before / after
- pass rate {b.mean.estimate:.1%} -> {f.mean.estimate:.1%}; pass^3 {b.pass_hat_k:.1%} -> {f.pass_hat_k:.1%}
- paired: {paired}

## Release gate
{gate.to_markdown()}

## Risks and next steps
- dropped_request and generic_fallback remain; the dropped-request judge is not trusted yet.
- Re-measure live (gpt-oss-20b) before release; record a new baseline only after approval.
"""
    path.write_text(text, encoding="utf-8")
    return text
```

</details>

In [ ]:
# Step 7: the release report your lead reads.
def write_report(path):

    # TODO 5: write the release report from the measured objects (see the hint above this cell).
    path.write_text("", encoding="utf-8")
    return ""


report_path = OUT_DIR / "project10-release-report.md"
report_text = write_report(report_path)
print(report_text[:1500] or "(empty: TODO 5)")

**You should now see** the start of the report, beginning `# Release report` and `**SHIP.**` (solution). Saved to `data/outputs/m10/project10-release-report.md`.

---
## Step 8 · 💸 Optional: re-measure on a real model

> 💸 Calls `gpt-oss-20b` (≈ 150–200 calls). Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

Offline numbers come from a rule-based model. Before a real release, re-run a slice on the production model and look at the **worst** run, not only the mean.

In [ ]:
# 💸 Optional: re-measure on gpt-oss-20b (mean and worst of 3 runs, shipped vs fixed).
LIVE_CASES = 10
if HAVE_GROQ_KEY:
    from mastery.agents.release_suite import release_cases
    from mastery.models import get_model

    def live_model(role):
        return get_model("fast", temperature=0, reasoning_effort="low")

    for cand in (BASELINE, FIXED):
        res = run_suite(cand, release_cases()[:LIVE_CASES], repetitions=3, redteam=[], model_for=live_model)
        means = [sum(r["passed"] for r in run.values()) / len(run) for run in res["runs"]]
        print(f"{cand.name:<12} mean {sum(means) / 3:.0%}  worst run {min(means):.0%}  calls/case {res['calls_per_case']:.2f}")
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

---
## Acceptance checks

In [ ]:
# Acceptance checks (from the spec). ✅ / ❌ per check. Each one fails on the untouched starter.
import inspect

from mastery.agents.release_suite import _ID

def uses(fn, *names):
    return bool(set(names) & set(fn.__code__.co_names))


reviewer = {t["id"]: reference_label(t) for t in raw}
agree = sum(labels[i] == reviewer[i] for i in reviewer) / len(reviewer)
failures = [r for r in sheet.rows if r["category"] not in ("", "pass")]

# The ranking an independent reviewer's labels give with YOUR severities.
ref_sheet = LabelSheet([{"trace_id": i, "category": c} for i, c in reviewer.items()])
ref_top = [r.name for r in TAXONOMY.rank(ref_sheet)[:2]] if TAXONOMY.categories else ["?"]

# Judge numbers recomputed here on judge-test with your labels.
_, _test = split_dev_test(traces, key="id")
expected = {}
for c, j in judges.items():
    flags = [bool(j(**dict(zip(("inputs", "outputs"), judge_view(t))))["score"]) for t in _test]
    expected[c] = agreement(flags, [labels[t["id"]] == c for t in _test])
rule = {c: a.tnr.low >= 0.85 and a.tpr.estimate >= 0.8 for c, a in expected.items()}

# A held-out suite the fix has never seen (same generator, other seed).
holdout = []
for item in generate_inputs(120, seed=4242, start=9001):
    if item["kind"] != "adversarial":
        m = _ID.search(item["turns"][0])
        holdout.append({**item, "signed_in": m.group(0).upper() if m else None})
_, h_before = evaluate_release(BASELINE, shipped, cases=holdout)
_, h_after = evaluate_release(FIXED, shipped, cases=holdout)


def halved(b, a):
    return bool(TOP_TWO) and all(label_counts(b)[c] > 0 and label_counts(a)[c] <= label_counts(b)[c] / 2 for c in TOP_TWO)


grew = [c for c, n in label_counts(after).items() if n > label_counts(before).get(c, 0)]


def section(name):
    found = re.search(rf"^##\s+{name}\b(.*?)(?=^##\s|\Z)", report_text, flags=re.M | re.S)
    return found.group(1).strip() if found else ""


SECTIONS = ("Decision", "Error analysis", "Judges", "Before / after", "Release gate", "Risks and next steps")
checks = {
    "1 ≥ 100 traces labelled, every failure with a note":
        len(sheet.labelled()) >= 100 and bool(failures) and all(len(r["note"]) >= 10 for r in failures),
    "2 labels agree ≥ 90% with an independent reviewer, without reading the answer key":
        agree >= 0.90 and not uses(label_trace, "reference_label", "reference_note")
        and '"want"' not in inspect.getsource(label_trace) and "'want'" not in inspect.getsource(label_trace),
    "3 taxonomy defines every label you used (validate() finds no problem)":
        bool(TAXONOMY.categories) and TAXONOMY.validate(sheet, min_categories=4) == [],
    "4 top two match the ranking of the reviewer's labels":
        len(TOP_TWO) == 2 and TOP_TWO == ref_top,
    "5 judge TPR/TNR measured on judge-test only, and the trust decisions follow the rule":
        set(judge_report) == set(expected) and all(
            (judge_report[c].tp, judge_report[c].fp, judge_report[c].tn, judge_report[c].fn)
            == (expected[c].tp, expected[c].fp, expected[c].tn, expected[c].fn) for c in expected)
        and TRUST == rule,
    "6 the fix halves each top-two category on the release suite AND on a held-out suite":
        halved(before, after) and halved(h_before, h_after),
    "7 the fix wins the paired comparison and adds no new failure (red team intact)":
        paired.significant and paired.b_wins > paired.a_wins and not grew and all(after["redteam"].values()),
    "8 the gate ships the fix and blocks the build without auth context":
        gate.passed and not fixture_gate.passed,
    "9 the report has all six sections, the decision, and the measured pass^3 numbers":
        all(section(s) for s in SECTIONS) and re.search(r"\b(SHIP|BLOCK)\b", section("Decision")) is not None
        and f"{before['summary'].pass_hat_k:.1%}" in report_text and f"{after['summary'].pass_hat_k:.1%}" in report_text
        and ("SHIP" in section("Decision")) == (gate.passed and not fixture_gate.passed),
}
for name, ok in checks.items():
    print("✅" if ok else "❌", name)
print(f"\n{sum(checks.values())}/{len(checks)} acceptance checks passed")
_ = (OUT_DIR / "project10-acceptance.json").write_text(json.dumps(checks, indent=2), encoding="utf-8")

**You should now see** `9/9 acceptance checks passed` (solution). Starter: `0/9`. The checks can't be passed with shortcuts: labels are compared with an independent reviewer and must not read the answer key, the top two are recomputed from the reviewer's labels, judge numbers are recomputed on judge-test, the fix is also measured on a held-out suite it has never seen, and the report must carry the measured numbers and a decision that matches the gates. The result is saved to `data/outputs/m10/project10-acceptance.json`.

## Findings (write yours)

- Which failure surprised you most while reading traces, and would a dashboard have shown it?
- What would make the dropped-request judge trustworthy? (Read its false positives on judge-**dev**.)
- After approval, record the new baseline (`python -m mastery.evals.release_gate --write-baseline`, Lesson 10.08) with your fix merged, and log the decision in [`docs/decision-log.md`](../../../docs/decision-log.md).

## ✅ Before you commit

- [ ] Restart & Run All passes without a key.
- [ ] All TODOs done; 9/9 acceptance checks; report and findings written.
- [ ] Outputs cleared (`jupyter nbconvert --clear-output --inplace <notebook>`); `data/outputs/` not committed.

---
## ➡️ What's next

🎉 **Project 10 done:** you ran a release the way a production team does: error analysis, validated judges, a measured fix and a gate that blocks bad builds.

**Next:** **Module 11 · Deep Agents** (see [COURSE_OUTLINE.md → Module 11](../../../COURSE_OUTLINE.md#module-11--deep-agents)): the same eval workflow, applied to an agent with planning, a virtual filesystem and subagents. Back to the [Module 10 README](../../../modules/10-testing-evaluation-observability/README.md).